# Coding Assistant Evaluations — Notebook 01: Intro & Approach

This workshop scores three coding agents — **Claude Code**, **Kiro**, and a **custom agent you build** — on a curated, prebuilt task set, along **two axes**:

1. **Pair-programmer** — when you ask a question about the codebase, does the agent surface the right files (an information-retrieval problem) and answer correctly?
2. **Autonomous** — given a task, does it produce a mergeable diff reliably?

## How this workshop is structured

The eval data is **prebuilt and curated**, not LLM-generated. That matters because if Claude writes the tasks *and* gets graded on them, you measure self-preference, not capability. We ship:

- A 9-task `tasks.yaml` grounded in real files in [`aws-samples/sample-agentic-platform`](https://github.com/aws-samples/sample-agentic-platform) at a pinned SHA.
- Per-task rubrics under `scaffolding/ground_truth/`.
- A hand-authored gold-standard set under `scaffolding/gold_standard/` for calibrating the LLM judge.

You spend the first few notebooks **inspecting** that data so you understand the schema and trust the methodology. Then you fill in a **custom-agent harness skeleton** at `my_agent/` (CLI plumbing already done — you implement the agent loop and tools), and run the eval against all three agents.

By the end you will have:

1. A reproducible 9-task eval set (5 normal + 2 trap + 2 nav-only).
2. Calibrated rubrics: 7 ground-truth rubrics + 8 hand-authored gold-standard pairs proving the LLM judge agrees with humans ≥ 80 % of the time.
3. A working custom Strands-based agent that you wired up.
4. **Pair-programmer scorecard** — precision@5, recall@10, MRR, answer accuracy, citation grounding, honesty.
5. **Autonomous scorecard** — pass rates by difficulty, reliability across 3 seeds on the hardest tasks, sequence-aware tool-call quality, wall-clock efficiency.
6. A reusable PR-review CI workflow — same reviewer, dropped into GitHub Actions.

## Why not just use SWE-Bench?

Public benchmarks are fine for vendor comparison but wrong for deciding whether an agent will work on **your** codebase. Two reasons:

- **Relevance**: the tasks don't look like your code, use your MCP tooling, or follow your review standards.
- **Contamination**: SWE-Bench tasks come from high-traffic public repos. Model providers almost certainly have the issues, PRs, and discussion in training data. Strong scores mix capability with leakage at an unknowable ratio.

This workshop uses a public repo (`aws-samples/sample-agentic-platform`) as a stand-in so everyone can follow along, but the whole structure generalizes: swap the repo URL in `tasks.yaml`, hand-curate new tasks following the schema docs in notebooks 02 and 03, run the eval.

## Pair-programmer metrics (notebook 06)

| Metric | What it asks |
|---|---|
| **precision@5** | Of the first 5 files the agent touched, how many were relevant? |
| **recall@10** | Of the relevant files, how many surfaced in the first 10? |
| **MRR** | How quickly did the first relevant file appear? |
| **answer accuracy** | LLM judge against your ground-truth answer |
| **citation grounding** | All `path:line` refs in the answer actually exist |
| **honesty** | On trap tasks, did the agent refuse to fabricate a fix? |

## Autonomous metrics (notebook 07)

| Signal | What it measures |
|---|---|
| **Rubric review** | Does the produced PR meet our review standards? |
| **Tests** | Does it still work? |
| **Static** | Does it pass the repo's linters? |
| **Tool-call (sequence-aware)** | Right tools, called *before* the edit, results consumed? |
| **Reliability** | Pass-rate across 3 seeds on the hardest tasks |
| **Wall-clock efficiency** | Uniform across all 3 agents (Kiro can't be intercepted for tokens) |

## How the notebooks work

Two phases:

**Phase 1 — Inspect the prebuilt data (notebooks 02-04).** You read the curated tasks, rubrics, and gold-standard, validate them with the included validators, and learn the schema in case you want to adapt the workshop to your own repo. No code-writing.

**Phase 2 — Run the eval (notebooks 05-07).** You fill in the `my_agent/` harness skeleton, then run the pair-programmer and autonomous evals against all three agents.

## Module layout

| | |
|---|---|
| **01** (this) | Why + how + env check |
| **02** | Inspect the prebuilt task set + schema reference |
| **03** | Inspect the prebuilt rubrics + gold standard + schema reference |
| **04** | Calibrate the automated PR reviewer against the gold set |
| **05** | Fill in the `my_agent/` harness skeleton (CLI plumbing already done) |
| **06** | Pair-programmer eval — IR + correctness + grounding + honesty |
| **07** | Autonomous eval + reliability sub-study + final report |

## Prereqs

Notebook 01 (this one) is the only one with a meaningful code cell — it verifies your environment. Notebooks 02-04 just read prebuilt data. Notebook 05 is where you'll start writing code.

In [ ]:
%pip install -q -r requirements.txt

### Environment smoke test

Verifies: AWS creds, Bedrock model access, `claude` CLI, `kiro-cli`, `uv` and `git`.

If `kiro-cli` is missing, you can still do the workshop — just exclude it from the final run in notebook 07.

> If `claude` shows FAIL but you know it's installed, your Jupyter kernel's `PATH` doesn't include `~/.local/bin` (or wherever the binary lives). Restart the kernel from a shell that has it on `PATH`, or set `os.environ['PATH']` in this notebook before the check.

In [ ]:
import os, shutil, boto3

# Make sure user-local bins are on PATH for the kernel — Jupyter often
# launches with a stripped-down PATH that misses ~/.local/bin etc.
for p in (os.path.expanduser('~/.local/bin'), '/opt/homebrew/bin', '/usr/local/bin'):
    if p not in os.environ.get('PATH', '').split(':') and os.path.isdir(p):
        os.environ['PATH'] = p + ':' + os.environ.get('PATH', '')

MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

def check(label, ok, detail=''):
    mark = 'OK  ' if ok else 'FAIL'
    print(f'{mark}  {label}' + (f'  ({detail})' if detail else ''))

try:
    who = boto3.client('sts').get_caller_identity()
    check('AWS credentials', True, who['Arn'])
except Exception as e:
    check('AWS credentials', False, str(e))

try:
    rt = boto3.client('bedrock-runtime', region_name='us-east-1')
    resp = rt.converse(
        modelId=MODEL_ID,
        messages=[{'role': 'user', 'content': [{'text': 'Say hi in one word.'}]}],
        inferenceConfig={'maxTokens': 10},
    )
    check('Bedrock Claude Sonnet 4.5', True, resp['output']['message']['content'][0]['text'])
except Exception as e:
    check('Bedrock Claude Sonnet 4.5', False, str(e))

# `kiro-cli` is the binary name; the `kiro` shell alias only works in
# interactive shells, not subprocess calls.
for tool in ('claude', 'kiro-cli', 'uv', 'git'):
    path = shutil.which(tool)
    check(f'`{tool}` on PATH', bool(path), path or 'not found')

### What you need next

- **Optional**: A terminal open to a Claude Code (or Kiro) session if you want help filling in `my_agent/` in notebook 05. Not required — the skeleton + TODOs are enough to do it by hand.
- This notebook stays open; the next six will do the rest.

Move on to **`02 inspect prebuilt tasks.ipynb`**.